# TestingAI - Комплексная оценка нейронных сетей для дефектоскопии труб

## Полный цикл оценки моделей:
1. **Загрузка и подготовка данных** из .raw файлов
2. **Обучение модели** с оптимизацией по валидационной выборке
3. **Генерация предсказаний** на тестовой выборке (метки классов и вероятности)
4. **Расчёт базовых метрик**: Accuracy, Precision, Recall, F1-Score, ROC-AUC
5. **Анализ матрицы ошибок** с макро- и взвешенными усреднениями
6. **Статистическая оценка надёжности** (95% доверительные интервалы методом бутстрапа)
7. **Оценка вычислительных характеристик** (время обучения, RAM/VRAM, инференс)
8. **Анализ масштабируемости** и зависимости от размера батча
9. **Анализ кривых обучения** (зависимость от размера выборки)

In [1]:
# Import all required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import struct
from pathlib import Path
import math
import json
from datetime import datetime
import os
import time
import traceback
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_recall_fscore_support, roc_curve
from sklearn.model_selection import train_test_split
import pickle

# CUDA optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

def log_message(msg):
    output_dir="model_experiments"
    output_path = Path(output_dir)
    log_file = output_path / 'logs' / 'training_log.txt'
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_entry = f"[{timestamp}] {msg}"
    print(log_entry)
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(log_entry + '\n')
    

def read_raw_fileOLD(filepath):
    """Чтение .raw файла с парсингом заголовка и матрицы данных (аналогично PipeTransformer.ipynb)"""
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    with open(path, 'rb') as f:
        data = f.read()
    
    # Format: 16-byte header + x*y doubles
    # 0-3: x (int), 4-7: y (int), 8-15: thicknom (double), 16+: matrix data
    if len(data) < 16:
        raise ValueError('File too short')
    
    x = struct.unpack_from('>i', data, 0)[0]
    y = struct.unpack_from('>i', data, 4)[0]
    thicknom = struct.unpack_from('>d', data, 8)[0]
    
    # Data starts at offset 16
    num_values = x * y
    if len(data) < 16 + num_values * 8:
        raise ValueError(f'File too short for {x}x{y} matrix')
    matrix = struct.unpack(f'>{num_values}d', data[16:16+num_values*8])
    matrix_np = np.array(matrix, dtype=np.float64).reshape(x, y).astype(np.float32)
    
    # Determine if defective based on filename or thicknom value
    defective = 1 if 'defect' in path.name.lower() or thicknom < 0 else 0
    
    return {'matrix': matrix_np, 'defective': defective, 'thicknom': thicknom}


def read_raw_file(filepath):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    with open(path, 'rb') as f:
        data = f.read()
    if len(data) < 17:
        raise ValueError('File too short')
    x = struct.unpack_from('>i', data, 0)[0]
    y = struct.unpack_from('>i', data, 4)[0]
    thicknom = struct.unpack_from('>d', data, 8)[0]
    defective = struct.unpack_from('>B', data, 16)[0]
    num_values = x * y
    matrix = struct.unpack(f'>{num_values}d', data[17:17+num_values*8])
    matrix_np = np.array(matrix, dtype=np.float32).reshape(x, y)
    return {'matrix': matrix_np, 'defective': defective, 'thicknom': thicknom}


def load_dataset_from_folderOLD(folder_path, verbose=True):
    """Загрузка всех .raw файлов из папки с метками (аналогично PipeTransformer.ipynb)"""
    folder = Path(folder_path)
    raw_files = list(folder.glob('*.raw'))
    if not raw_files:
        raise FileNotFoundError(f'No .raw files found in {folder_path}')
    if verbose:
        print(f'Found files: {len(raw_files)}')
    data_list, labels, thicknom_list = [], [], []
    for file_path in raw_files:
        try:
            result = read_raw_file(file_path)
            data_list.append(result['matrix'])
            labels.append(result['defective'])
            thicknom_list.append(result['thicknom'])
        except Exception as e:
            print(f'Ошибка чтения {file_path.name}: {e}')
    if verbose:
        print(f'Loaded: {len(data_list)} files')
    return data_list, np.array(labels), thicknom_list

def load_dataset_from_folderOLD2(folder_path, verbose=True):
    """Загружает все .raw файлы из папки"""
    folder = Path(folder_path)
    raw_files = list(folder.glob('*.raw'))
    
    if not raw_files:
        raise FileNotFoundError(f"❌ Не найдено .raw файлов в {folder_path}")
    
    if verbose:
        print(f"📁 Найдено файлов: {len(raw_files)}")
    
    data_list = []
    labels = []
   # filenames = []
    thicknom_list = []
    
    good_count = 0
    bad_count = 0
    
    for file_path in raw_files:
        try:
            result = read_raw_file(file_path)
            data_list.append(result['matrix'])
            labels.append(result['defective'])
            #filenames.append(result['filename'])
            thicknom_list.append(result['thicknom'])
            print(result['defective'])
            if result['defective'] == 0:
                good_count += 1
            else:
                bad_count += 1
                
            if verbose and len(data_list) % 10 == 0:
                print(f"  Загружено: {len(data_list)}/{len(raw_files)}")
                
        except Exception as e:
            print(f"⚠️ Ошибка при чтении {file_path.name}: {e}")
            continue
    
    if verbose:
        print(f"\n✅ Загружено: {len(data_list)} файлов")
        print(f"   🟢 Годных (0): {good_count}")
        print(f"   🔴 Бракованных (1): {bad_count}")
        print(f"   📊 Размер матрицы: {data_list[0].shape}")
    
    return data_list, np.array(labels), thicknom_list

def load_dataset_from_folder(folder_path, verbose=True):
    folder = Path(folder_path)
    raw_files = list(folder.glob('*.raw'))
    if not raw_files:
        raise FileNotFoundError(f'No .raw files found in {folder_path}')
    if verbose:
        print(f'Found files: {len(raw_files)}')
    data_list, labels, thicknom_list = [], [], []
    for file_path in raw_files:
        try:
            result = read_raw_file(file_path)
            data_list.append(result['matrix'])
            labels.append(result['defective'])
            #print(result['defective'])
            thicknom_list.append(result['thicknom'])
        except Exception as e:
            print(f'Error reading {file_path.name}: {e}')
    if verbose:
        print(f'Loaded: {len(data_list)} files')
    return data_list, np.array(labels), thicknom_list

def prepare_and_split_data(X, y, test_size=0.15, val_size=0.15, random_state=42, apply_augmentation=False):
    """
    Подготовка данных: масштабирование, аугментация и разделение на train/val/test.
    Аналогично обработке в PipeTransformer.ipynb.
    
    Args:
        X: массив данных形状 (n_samples, n_channels, n_timepoints) или список матриц
        y: метки классов
        test_size: доля тестовой выборки
        val_size: доля валидационной выборки
        random_state: seed для воспроизводимости
        apply_augmentation: применять ли аугментацию данных
    
    Returns:
        dict с ключами: 'X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test'
    """
    np.random.seed(random_state)
    
    # Convert list of matrices to numpy array if needed
    if isinstance(X, list):
        max_len = max([m.shape[0] for m in X])
        n_channels = X[0].shape[1] if len(X[0].shape) > 1 else 1
        X = np.array([np.pad(m, ((0, max_len - m.shape[0]), (0, 0)), mode='constant') 
                      for m in X])
    
    n_samples, n_channels, n_timepoints = X.shape
    
    # Reshape for scaler: (n_samples, n_channels * n_timepoints)
    X_flat = X.reshape(n_samples, -1)
    
    # Replace inf and nan with finite values
    X_flat = np.nan_to_num(X_flat, nan=0.0, posinf=1e10, neginf=-1e10)
    
    # Scale data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_flat).reshape(n_samples, n_channels, n_timepoints)

    # === КЛЮЧЕВОЕ ИЗМЕНЕНИЕ: Транспонируем для формата (n_samples, n_timepoints, n_channels) ===
    # Это нужно для корректной работы collate_fn и моделей
    # Вход в модели ожидает (Batch, Length, Channels), поэтому транспонируем
    X_scaled = X_scaled.transpose(0, 2, 1)
    
    # Split data: first to train+test, then train to train+val
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # Calculate val_size relative to train_val set
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_size_adjusted, 
        random_state=random_state, stratify=y_train_val
    )
    
    # Apply augmentation if requested
    if apply_augmentation:
        X_train_aug, y_train_aug = augment_data(X_train, y_train, random_state=random_state)
        X_train = np.concatenate([X_train, X_train_aug], axis=0)
        y_train = np.concatenate([y_train, y_train_aug], axis=0)
    
    return {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test
    }

def augment_data(X, y, augmentation_factor=2, random_state=42):
    """
    Аугментация данных добавлением гауссовского шума.
    """
    np.random.seed(random_state)
    n_aug = len(X) * augmentation_factor
    noise_level = 0.1
    
    X_aug = []
    y_aug = []
    
    for i in range(n_aug):
        idx = np.random.randint(len(X))
        sample = X[idx].copy()
        noise = np.random.normal(0, noise_level, sample.shape).astype(np.float32)
        sample = sample + noise
        X_aug.append(sample)
        y_aug.append(y[idx])
    
    return np.array(X_aug), np.array(y_aug)


class PipeDataset(Dataset):
    """Dataset для обработки данных с масштабированием и паддингом (аналогично PipeTransformer.ipynb)"""
    def __init__(self, data_list, labels, scaler=None, fit_scaler=False):
        self.data_list = data_list
        self.labels = labels
        if fit_scaler:
            all_data = np.vstack(data_list)
            self.scaler = StandardScaler().fit(all_data)
        else:
            self.scaler = scaler
        if self.scaler is not None:
            self.data_list = [torch.FloatTensor(self.scaler.transform(sample)) for sample in self.data_list]
        else:
            self.data_list = [torch.FloatTensor(sample) for sample in self.data_list]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        return self.data_list[idx], self.labels[idx]

def collate_fn(batch):
    """Функция для создания батча с паддингом (аналогично PipeTransformer.ipynb)"""
    data_list, labels = zip(*batch)
    max_len = max([d.shape[0] for d in data_list])
    padded_data, lengths = [], []
    for data in data_list:
        length = data.shape[0]
        lengths.append(length)
        padding = torch.zeros(max_len - length, data.shape[1])
        padded = torch.cat([data, padding], dim=0)
        padded_data.append(padded)
    data_batch = torch.stack(padded_data)
    labels_batch = torch.stack(labels)
    mask = torch.zeros(data_batch.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1
    return data_batch, labels_batch, torch.tensor(lengths), mask


Using device: cuda


6 Модели

In [2]:

class Pipe1DCNN(nn.Module):
    def __init__(self, input_channels=8, n_classes=3, dropout=0.3):
        super().__init__()

        self.features = nn.Sequential(
            # Блок 1
            nn.Conv1d(input_channels, 32, kernel_size=5, padding=2),  # padding='same'
            nn.InstanceNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Уменьшаем длину в 2 раза

            # Блок 2
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.InstanceNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Уменьшаем длину в 2 раза

            # Блок 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.InstanceNorm1d(128),
            nn.ReLU(),

            # АДАПТИВНЫЙ ПУЛИНГ - ключ к переменной длине!
            # Превращает (Batch, 128, Any_Length) -> (Batch, 128, 1)
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x, mask=None):
        # Вход: (Batch, Length, 8) -> для Conv1d нужно (Batch, 8, Length)
        x = x.transpose(1, 2)
        x = self.features(x)  # (Batch, 128, 1)
        x = self.classifier(x)
        return x


class ImprovedPipeCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(8, 32, 9, padding=4),
            nn.InstanceNorm1d(32),
            nn.ReLU()
        )

        self.block1 = ResidualBlock(32)
        self.block2 = ResidualBlock(32)

        self.down = nn.Conv1d(32, 64, 3, stride=2, padding=1)

        self.block3 = ResidualBlock(64)
        self.block4 = ResidualBlock(64)

        self.attention = nn.Sequential(
            nn.Conv1d(64, 32, 1),
            nn.ReLU(),
            nn.Conv1d(32, 1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = x.transpose(1, 2)

        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)

        x = self.down(x)

        x = self.block3(x)
        x = self.block4(x)

        w = self.attention(x)
        w = torch.softmax(w, dim=2)

        x = (x * w).sum(dim=2)

        return self.classifier(x)
    

class PipeCNNLSTM(nn.Module):
    def __init__(self, input_channels=8, n_classes=2, 
                 cnn_filters=[32, 64, 128], 
                 lstm_hidden=128, 
                 lstm_layers=4,
                 dropout=0.3):
        super().__init__()
        
        # === CNN БЛОК (извлекает локальные признаки) ===
        cnn_layers = []
        in_channels = input_channels
        
        for filters in cnn_filters:
            cnn_layers.extend([
                nn.Conv1d(in_channels, filters, kernel_size=5, padding=2),
                nn.InstanceNorm1d(filters),
                nn.ReLU(),
                nn.MaxPool1d(2),  
                nn.Dropout(dropout)
            ])
            in_channels = filters
        
        self.cnn = nn.Sequential(*cnn_layers)
        
        # LSTM БЛОК 
        self.lstm = nn.LSTM(
            input_size=cnn_filters[-1],
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,  # Двусторонний LSTM
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        # === Классификатор ===
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),  # *2 для bidirectional
            #nn.ReLU(),
            nn.LeakyReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    
    def forward(self, x, lengths=None):
        # x: (B, L, C)
    
        # === CNN ===
        x = x.transpose(1, 2)
        x = self.cnn(x)
        x = x.transpose(1, 2)
    
        # === корректировка длин ===
        if lengths is not None:
            lengths = lengths.clone()
    
            num_pools = len(cnn_filters)

            for _ in range(num_pools):  # число MaxPool
                lengths = (lengths + 1) // 2
    
            lengths = torch.clamp(lengths, max=x.shape[1])
    
        # === PACK ===
        if lengths is not None:
            lengths_cpu = lengths.cpu()
    
            packed = pack_padded_sequence(
                x,
                lengths_cpu,
                batch_first=True,
                enforce_sorted=False
            )
    
            packed_out, (h_n, _) = self.lstm(packed)
    
            lstm_out, _ = pad_packed_sequence(
                packed_out,
                batch_first=True
            )
        else:
            lstm_out, (h_n, _) = self.lstm(x)
    
        # === ВЫХОД ===
        if self.lstm.bidirectional:
            h_cat = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_cat = h_n[-1]
    
        return self.classifier(h_cat)


## 7. Функции метрик и визуализации

In [3]:
def plot_confusion_matrix(y_true, y_pred, class_names=None, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names or ['Class 0', 'Class 1'],
                yticklabels=class_names or ['Class 0', 'Class 1'])
    plt.title('Матрица ошибок (Confusion Matrix)', fontsize=14)
    plt.xlabel('Предсказано', fontsize=12)
    plt.ylabel('Истинное', fontsize=12)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Матрица ошибок сохранена: {save_path}")
    else:
        plt.show()
    plt.close()
    return cm

def plot_roc_curve(y_true, y_probs, save_path=None):
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    auc = roc_auc_score(y_true, y_probs)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC-кривая', fontsize=14)
    plt.legend(loc="lower right")
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 ROC-кривая сохранена: {save_path}")
    else:
        plt.show()
    plt.close()
    return fpr, tpr, auc

def bootstrap_confidence_interval(y_true, y_pred, metric_func, n_bootstrap=1000, confidence=0.95):
    """Расчёт доверительного интервала методом бутстрапа"""
    np.random.seed(42)
    n_samples = len(y_true)
    bootstrap_scores = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        score = metric_func(y_true[indices], y_pred[indices])
        bootstrap_scores.append(score)
    lower = np.percentile(bootstrap_scores, (1 - confidence) / 2 * 100)
    upper = np.percentile(bootstrap_scores, (1 + confidence) / 2 * 100)
    mean = np.mean(bootstrap_scores)
    return mean, lower, upper

def calculate_all_metrics(y_true, y_pred, y_probs):
    """Расчёт всех базовых метрик с макро- и взвешенными усреднениями"""
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    precision_binary, recall_binary, f1_binary, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0)
    auc_roc = roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else 0.5
    
    metrics = {
        'accuracy': accuracy,
        'precision_binary': precision_binary, 'recall_binary': recall_binary, 'f1_binary': f1_binary,
        'precision_macro': precision_macro, 'recall_macro': recall_macro, 'f1_macro': f1_macro,
        'precision_weighted': precision_weighted, 'recall_weighted': recall_weighted, 'f1_weighted': f1_weighted,
        'auc_roc': auc_roc
    }
    return metrics

## 8. Мониторинг памяти и вычислительных ресурсов

In [4]:
class ResourceMonitor:
    """Мониторинг RAM и VRAM"""
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.peak_ram = 0
        self.peak_vram = 0
    
    def get_ram_usage(self):
        ram_mb = self.process.memory_info().rss / 1024 / 1024
        self.peak_ram = max(self.peak_ram, ram_mb)
        return ram_mb
    
    def get_vram_usage(self):
        if torch.cuda.is_available():
            vram_mb = torch.cuda.memory_allocated() / 1024 / 1024
            self.peak_vram = max(self.peak_vram, vram_mb)
            return vram_mb
        return 0
    
    def reset_peak(self):
        self.peak_ram = 0
        self.peak_vram = 0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    
    def get_peak_stats(self):
        stats = {'peak_ram_mb': self.peak_ram, 'peak_vram_mb': self.peak_vram}
        if torch.cuda.is_available():
            stats['peak_vram_mb'] = torch.cuda.max_memory_allocated() / 1024 / 1024
        return stats

monitor = ResourceMonitor()

## 9. Обучение модели с трекингом ресурсов

In [5]:
def train_with_tracking(model, train_loader, val_loader, epochs=50, lr=0.001, device='cpu', patience=10):
    """Обучение модели с мониторингом времени и памяти"""
    monitor.reset_peak()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    
    start_time = time.time()
    epoch_times = []
    
    print(f"🚀 Начало обучения на {device}")
    print(f"📊 Параметры модели: {count_parameters(model):,}")
    
    for epoch in range(epochs):
        epoch_start = time.time()
        
        # Train
        model.train()
        total_loss, correct, total = 0, 0, 0
        for data, labels, mask, lengths in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(data, lengths)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(data, lengths)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        train_loss = total_loss / total
        train_acc = 100. * correct / total
        
        # Validate
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for data, labels, mask, lengths in val_loader:
                data, labels = data.to(device), labels.to(device)
                outputs = model(data, lengths)
                loss = criterion(outputs, labels)
                total_loss += loss.item() * data.size(0)
                total += labels.size(0)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
        val_loss = total_loss / total
        val_acc = 100. * correct / total
        
        scheduler.step()
        
        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {epoch_time:.2f}s")
        
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping на эпохе {epoch+1}")
            log_message(f"  train_with_tracking complit")
            break
    
    total_time = time.time() - start_time
    
    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
    
    training_stats = {
        'total_time_sec': total_time,
        'avg_epoch_time_sec': np.mean(epoch_times),
        'epochs_completed': len(history['train_loss']),
        'best_val_acc': best_val_acc,
        'peak_ram_mb': monitor.get_peak_stats()['peak_ram_mb'],
        'peak_vram_mb': monitor.get_peak_stats()['peak_vram_mb'],
        'n_parameters': count_parameters(model)
    }
    
    print(f"\n✅ Обучение завершено!")
    print(f"   Время обучения: {total_time:.2f} сек ({total_time/60:.2f} мин)")
    print(f"   Лучшая точность на валидации: {best_val_acc:.2f}%")
    print(f"   Пиковое потребление RAM: {training_stats['peak_ram_mb']:.2f} MB")
    print(f"   Пиковое потребление VRAM: {training_stats['peak_vram_mb']:.2f} MB")
    #log_message("=" * 80)
    log_message(f"\n✅ Обучение завершено!")
    log_message(f"   Время обучения: {total_time:.2f} сек ({total_time/60:.2f} мин)")
    log_message(f"   Лучшая точность на валидации: {best_val_acc:.2f}%")
    log_message(f"   Пиковое потребление RAM: {training_stats['peak_ram_mb']:.2f} MB")
    log_message(f"   Пиковое потребление VRAM: {training_stats['peak_vram_mb']:.2f} MB")
    
    return model, history, training_stats

## 10. Бенчмаркинг инференса

In [6]:
def benchmark_inference(model, test_loader, batch_sizes=[8, 16, 32], device='cpu', n_warmup=10, n_runs=100):
    """
    Измерение характеристик инференса для разных размеров батча.
    """
    model.eval()
    results = []
    
    print("="*60)
    print("🔬 БЕНЧМАРКИНГ ИНФЕРЕНСА")
    print("="*60)
    
    for bs in batch_sizes:
        # Create a subset loader with specific batch size
        test_dataset = PipeDataset(
            [test_loader.dataset.data_list[i] for i in range(min(100, len(test_loader.dataset)))],
            [test_loader.dataset.labels[i].item() for i in range(min(100, len(test_loader.dataset)))]
        )
        subset_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, collate_fn=collate_fn)
        
        # Warmup
        for _ in range(n_warmup):
            for data, labels, mask, lengths in subset_loader:
                data = data.to(device)
                with torch.no_grad():
                    _ = model(data, lengths)
        
        # Benchmark
        monitor.reset_peak()
        start_time = time.time()
        inference_count = 0
        
        with torch.no_grad():
            for _ in range(n_runs):
                for data, labels, mask, lengths in subset_loader:
                    data = data.to(device)
                    outputs = model(data, lengths)
                    inference_count += data.size(0)
        
        total_time = time.time() - start_time
        peak_stats = monitor.get_peak_stats()
        
        latency_ms = (total_time / inference_count) * 1000 * n_runs
        throughput = inference_count / total_time
        
        result = {
            'batch_size': bs,
            'total_time_sec': total_time,
            'inference_count': inference_count * n_runs,
            'latency_ms': latency_ms,
            'throughput_samples_per_sec': throughput,
            'peak_ram_mb': peak_stats['peak_ram_mb'],
            'peak_vram_mb': peak_stats['peak_vram_mb']
        }
        results.append(result)
        
        print(f"\nBatch Size: {bs}")
        print(f"   Время {n_runs} проходов: {total_time:.2f} сек")
        print(f"   Задержка (Latency): {latency_ms:.3f} мс/образец")
        print(f"   Пропускная способность: {throughput:.2f} образцов/сек")
        print(f"   Пик RAM: {result['peak_ram_mb']:.2f} MB")
        print(f"   Пик VRAM: {result['peak_vram_mb']:.2f} MB")
    
    return results

def analyze_sequence_length_dependency(model, data_dict, device='cpu'):
    """Анализ зависимости времени инференса от длины последовательности"""
    model.eval()
    
    # Group samples by length
    length_groups = {}
    for i, sample in enumerate(data_dict['X_test']):
        length = sample.shape[0]
        if length not in length_groups:
            length_groups[length] = []
        length_groups[length].append(i)
    
    results = []
    print("\n" + "="*60)
    print("📏 ЗАВИСИМОСТЬ ОТ ДЛИНЫ ПОСЛЕДОВАТЕЛЬНОСТИ")
    print("="*60)
    
    for length in sorted(length_groups.keys()):
        indices = length_groups[length][:10]  # Take up to 10 samples
        samples = [data_dict['X_test'][i] for i in indices]
        labels = [data_dict['y_test'][i] for i in indices]
        
        dataset = PipeDataset(samples, labels)
        loader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
        
        start_time = time.time()
        with torch.no_grad():
            for data, labels, mask, lengths in loader:
                data = data.to(device)
                _ = model(data, lengths)
        elapsed = time.time() - start_time
        
        avg_time = elapsed / len(indices) * 1000  # ms
        results.append({'length': length, 'avg_time_ms': avg_time})
        print(f"Длина {length:4d}: {avg_time:.3f} мс/образец")
    
    return results

## 11. Метрики эффективности и кривые обучения

In [17]:
def calculate_efficiency_metrics(accuracy, training_stats, inference_results):
    """
    Расчёт метрик эффективности:
    TrainingEfficiency = Accuracy / Training_time
    InferenceEfficiency = Accuracy / Inference_latency
    MemoryEfficiency = Accuracy / Peak_Memory_Usage
    """
    training_time = training_stats.get('total_time_sec', 1)
    inference_latency = inference_results[0]['latency_ms'] if inference_results else 1
    peak_memory = training_stats.get('peak_ram_mb', 1) + training_stats.get('peak_vram_mb', 1)
    
    efficiency = {
        'training_efficiency': accuracy / training_time if training_time > 0 else 0,
        'inference_efficiency': accuracy / inference_latency if inference_latency > 0 else 0,
        'memory_efficiency': accuracy / peak_memory if peak_memory > 0 else 0
    }
    return efficiency

def analyze_learning_curves(data_dict, model_class, device='cpu', 
                            sample_sizes=[100, 200, 400, 800, 1600],
                            n_repeats=3, epochs=30, random_state=42):
    """
    Анализ зависимости метрик от размера обучающей выборки.
    """
    np.random.seed(random_state)
    
    print("="*60)
    print("📈 АНАЛИЗ КРИВЫХ ОБУЧЕНИЯ")
    print("="*60)
    
    all_results = []
    
    for n_samples in sample_sizes:
        print(f"\n🔹 Размер выборки: {n_samples}")
        repeat_results = []
        
        for r in range(n_repeats):
            rs = random_state + r
            np.random.seed(rs)
            
            # Sample training data
            n_train = min(n_samples, len(data_dict['X_train']))
            indices = np.random.choice(len(data_dict['X_train']), size=n_train, replace=False)
            
            X_train_sub = [data_dict['X_train'][i] for i in indices]
            y_train_sub = data_dict['y_train'][indices]
            
            # Create loaders
            train_dataset = PipeDataset(X_train_sub, y_train_sub)
            val_dataset = PipeDataset(data_dict['X_val'], data_dict['y_val'])
            
            train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
            val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
            
            # Train model не используется
            model = model_class().to(device)
            _, history, stats = train_with_tracking(
                model, train_loader, val_loader, 
                epochs=epochs, lr=0.001, device=device, patience=10
            )
            
            # Evaluate on test
            test_dataset = PipeDataset(data_dict['X_test'], data_dict['y_test'])
            test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
            
            model.eval()
            all_preds, all_probs, all_labels = [], [], []
            with torch.no_grad():
                for data, labels, mask, lengths in test_loader:
                    data, labels = data.to(device), labels.to(device)
                    outputs = model(data, lengths)
                    probs = torch.softmax(outputs, dim=1)
                    _, predicted = outputs.max(1)
                    all_preds.extend(predicted.cpu().numpy())
                    all_probs.extend(probs[:, 1].cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
            
            metrics = calculate_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs))
            
            repeat_results.append({
                'n_samples': n_samples,
                'random_state': rs,
                'train_acc': history['train_acc'][-1] if history['train_acc'] else 0,
                'val_acc': history['val_acc'][-1] if history['val_acc'] else 0,
                'test_accuracy': metrics['accuracy'],
                'test_precision': metrics['precision_binary'],
                'test_recall': metrics['recall_binary'],
                'test_f1': metrics['f1_binary'],
                'test_auc': metrics['auc_roc'],
                'training_time': stats['total_time_sec'],
                'generalization_gap': history['train_acc'][-1] - metrics['accuracy'] if history['train_acc'] else 0
            })
        
        # Aggregate results
        agg = {
            'n_samples': n_samples,
            'test_accuracy_mean': np.mean([r['test_accuracy'] for r in repeat_results]),
            'test_accuracy_std': np.std([r['test_accuracy'] for r in repeat_results]),
            'test_f1_mean': np.mean([r['test_f1'] for r in repeat_results]),
            'test_f1_std': np.std([r['test_f1'] for r in repeat_results]),
            'training_time_mean': np.mean([r['training_time'] for r in repeat_results]),
            'generalization_gap_mean': np.mean([r['generalization_gap'] for r in repeat_results])
        }
        all_results.append(agg)
        
        print(f"   Accuracy: {agg['test_accuracy_mean']:.4f} ± {agg['test_accuracy_std']:.4f}")
        print(f"   F1-Score: {agg['test_f1_mean']:.4f} ± {agg['test_f1_std']:.4f}")
        print(f"   Время обучения: {agg['training_time_mean']:.2f} сек")
        print(f"   Разрыв обобщения: {agg['generalization_gap_mean']:.4f}")
    
    return all_results

def plot_learning_curve_analysis(results, save_path=None):
    """Построение графиков кривых обучения"""
    df = pd.DataFrame(results)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy vs log(N)
    axes[0, 0].errorbar(np.log(df['n_samples']), df['test_accuracy_mean'], 
                        yerr=df['test_accuracy_std'], capsize=5, marker='o', linestyle='-')
    axes[0, 0].set_xlabel('log(N) - логарифм размера выборки')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Зависимость точности от размера выборки')
    axes[0, 0].grid(True, alpha=0.3)
    
    # F1 vs log(N)
    axes[0, 1].errorbar(np.log(df['n_samples']), df['test_f1_mean'], 
                        yerr=df['test_f1_std'], capsize=5, marker='s', linestyle='--', color='orange')
    axes[0, 1].set_xlabel('log(N) - логарифм размера выборки')
    axes[0, 1].set_ylabel('F1-Score')
    axes[0, 1].set_title('Зависимость F1 от размера выборки')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Training time vs N
    axes[1, 0].plot(df['n_samples'], df['training_time_mean'], marker='^', linestyle='-', color='green')
    axes[1, 0].set_xlabel('Размер выборки (N)')
    axes[1, 0].set_ylabel('Время обучения (сек)')
    axes[1, 0].set_title('Зависимость времени обучения от размера выборки')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Generalization gap vs log(N)
    axes[1, 1].plot(np.log(df['n_samples']), df['generalization_gap_mean'], 
                    marker='d', linestyle='-.', color='red')
    axes[1, 1].set_xlabel('log(N) - логарифм размера выборки')
    axes[1, 1].set_ylabel('Разрыв обобщения (Train Acc - Test Acc)')
    axes[1, 1].set_title('Разрыв обобщения')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 График сохранён: {save_path}")
    else:
        plt.show()
    plt.close()
    
def evaluate(model, loader, criterion, device):
    log_message(" Start evaluate")
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs = []
    all_preds, all_labels = [], []
    log_message(" INIT evaluate")
    with torch.no_grad():
        for data, labels, lengths, mask in loader:
            data = data.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
            #with torch.amp.autocast():
                outputs = model(data, lengths)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * data.size(0)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    log_message(" Complit evaluate")
    
    # Calculate F1 score
    from sklearn.metrics import precision_recall_fscore_support
    _, _, f1_binary, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', zero_division=0)
    
    return total_loss / total, 100. * correct / total, all_preds, all_labels, f1_binary


In [19]:
# ============================================================================
## 12. МАСШТАБНОЕ ТЕСТИРОВАНИЕ МОДЕЛЕЙ С ПЕРЕБОРОМ ГИПЕРПАРАМЕТРОВ И АРХИТЕКТУР
# ============================================================================

def run_full_model_evaluation(base_data, output_dir="model_experiments", device="cuda"):
    """
    Полный цикл тестирования моделей с перебором гиперпараметров и архитектур.
    Сетка параметров:
    - sample_sizes: [100, 200, 400, 800, 1600]
    - batch_sizes: [8, 16, 32]
    - dropout: [0.2, 0.3, 0.4]
    - learning_rates: [0.001, 0.01, 0.05, 0.0001, 0.0005]
    - random_states: [42, 64, 31]
    - apply_augmentation: [True, False]
    - model_architectures: ['OneDCNN', 'PipeCNNLSTM', 'CNNWithClassifier', 'PipeTransformer']
    - epochs: 150, early_stop_patience: 50
    """
    sample_sizes = [100, 200, 400, 800, 1600]
    batch_sizes = [8, 16, 32]
    dropouts = [0.2, 0.3, 0.4]
    learning_rates = [0.001, 0.01, 0.05, 0.0001, 0.0005]
    random_states = [42, 64, 31]
    augmentations = [True, False]
    model_architectures = ['OneDCNN', 'PipeCNNLSTM', 'CNNWithClassifier', 'PipeTransformer']
    epochs = 150
    early_stop_patience = 10
    
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    (output_path / 'models').mkdir(exist_ok=True)
    (output_path / 'logs').mkdir(exist_ok=True)
    (output_path / 'results').mkdir(exist_ok=True)
    
    results_file = output_path / 'results' / 'all_results.json'
    
    checkpoint_file = output_path / 'results' / 'results_checkpoint.pkl'
    
    all_results = []
    completed_configs = set()
    
    criterion = nn.CrossEntropyLoss()
    
    if checkpoint_file.exists():
        with open(checkpoint_file, 'rb') as f:
            checkpoint = pickle.load(f)
            all_results = checkpoint.get('results', [])
            completed_configs = checkpoint.get('completed_configs', set())
            completed_configs = {tuple(c) if isinstance(c, list) else c for c in completed_configs}
        print(f"Загружено {len(all_results)} предыдущих результатов")
    
    
    total_configs = len(sample_sizes) * len(batch_sizes) * len(dropouts) * len(learning_rates) * len(random_states) * len(augmentations) * len(model_architectures)
    config_count = 0
    
    log_message("=" * 80)
    log_message(f"НАЧАЛО ТЕСТИРОВАНИЯ: {total_configs} конфигураций")
    log_message(f"Архитектуры: {model_architectures}")
    log_message("=" * 80)
    
    for n_samples in sample_sizes:
        for batch_size in batch_sizes:
            for dropout in dropouts:
                for lr in learning_rates:
                    for rs in random_states:
                        for apply_aug in augmentations:
                            for model_arch in model_architectures:
                                config_key = (n_samples, batch_size, dropout, lr, rs, apply_aug, model_arch)
                                config_count += 1
                                
                                if config_key in completed_configs:
                                    continue
                                
                                log_message(
                                    f"[{config_count}/{total_configs}] {model_arch}: "
                                    f"N={n_samples}, BS={batch_size}, DO={dropout}, "
                                    f"LR={lr}, RS={rs}, Aug={apply_aug}"
                                )
                                
                                try:
                                    data = prepare_and_split_data(
                                        base_data['X_full'], 
                                        base_data['y_full'], 
                                        test_size=0.15, 
                                        val_size=0.15, 
                                        random_state=rs, 
                                        apply_augmentation=apply_aug
                                    )
                                    
                                    if n_samples < len(data['X_train']):
                                        np.random.seed(rs)
                                        indices = np.random.choice(len(data['X_train']), n_samples, replace=False)
                                        X_train_sub = data['X_train'][indices]
                                        y_train_sub = data['y_train'][indices]
                                    else:
                                        X_train_sub = data['X_train']
                                        y_train_sub = data['y_train']
                                    
                                    train_dataset = PipeDataset(X_train_sub, y_train_sub)
                                    val_dataset = PipeDataset(data['X_val'], data['y_val'])
                                    test_dataset = PipeDataset(data['X_test'], data['y_test'])
                                    
                                    train_loader = DataLoader(
                                        train_dataset, 
                                        batch_size=batch_size, 
                                        shuffle=True,
                                        collate_fn=collate_fn,
                                        num_workers=0, pin_memory=False
                                    )
                                    val_loader = DataLoader(
                                        val_dataset, 
                                        batch_size=batch_size, 
                                        shuffle=False,
                                        collate_fn=collate_fn,
                                        num_workers=0, pin_memory=False
                                    )
                                    test_loader = DataLoader(
                                        test_dataset, 
                                        batch_size=batch_size, 
                                        shuffle=False,
                                        collate_fn=collate_fn,
                                        num_workers=0, pin_memory=False
                                    )
                                    
                                    # После транспонирования в prepare_and_split_data формат: (n_samples, n_timepoints, n_channels)
                                    n_channels = (
                                        X_train_sub.shape[2]
                                        if len(X_train_sub.shape) > 2
                                        else 1
                                    )
                                    
                                    # Инициализация модели в зависимости от архитектуры
                                    if model_arch == 'OneDCNN':
                                        model = Pipe1DCNN(input_channels=n_channels,  n_classes=2,  dropout=dropout ).to(device)
                                        #model = ImprovedPipeCNN(input_channels=n_channels,  n_classes=2,  dropout=dropout ).to(device)
                                    elif model_arch == 'PipeCNNLSTM':
                                        model = PipeCNNLSTM(
                                            input_channels=n_channels, 
                                            n_classes=2, 
                                            cnn_filters=[32, 64, 512, 128], 
                                            lstm_hidden=128, 
                                            lstm_layers=4, 
                                            dropout=dropout
                                        ).to(device)
                                    elif model_arch == 'CNNWithClassifier':
                                        model = PipeCNNEncoder(
                                            input_channels=n_channels, 
                                            n_classes=2, 
                                            dropout=dropout
                                        ).to(device)
                                    elif model_arch == 'PipeTransformer':
                                        model = PipeTransformer(
                                            input_channels=n_channels, 
                                            n_classes=2, 
                                            d_model=128, 
                                            nhead=8, 
                                            num_layers=12, 
                                            dim_feedforward=256, 
                                            dropout=dropout
                                        ).to(device)
                                    else:
                                        raise ValueError(f"Неизвестная архитектура: {model_arch}")
                                    
                                    n_params = count_parameters(model)
                                    log_message(f"  Параметры модели: {n_params:,}")
                                    
                                    model, history, training_stats = train_with_tracking(
                                        model, 
                                        train_loader, 
                                        val_loader, 
                                        epochs=epochs, 
                                        lr=lr, 
                                        device=device, 
                                        patience=early_stop_patience
                                    )
                                    log_message(f"  train_with_tracking complit")
                                    val_loss, val_acc, val_preds, val_labels, f1score = evaluate(
                                        model, 
                                        test_loader, 
                                        criterion,
                                        device=device
                                    )
                                    log_message(f"  eval_results init")
                                    inference_results = benchmark_inference(
                                        model, 
                                        test_loader, 
                                        batch_sizes=[8, 16, 32], 
                                        device=device
                                    )
                                    log_message(f"  inference_results init")
                                    efficiency_metrics = calculate_efficiency_metrics(
                                        val_acc, 
                                        training_stats, 
                                        inference_results
                                    )
                                    log_message(f"  efficiency_metrics init")
                                    model_filename = (
                                        f"{model_arch}_N{n_samples}_BS{batch_size}_"
                                        f"DO{dropout}_LR{lr}_RS{rs}_Aug{str(apply_aug).lower()}.pth"
                                    )
                                    log_message(f"  model_filename init")
                                    model_path = output_path / 'models' / model_filename
                                    torch.save({
                                        'model_state_dict': model.state_dict(),
                                        'config': {
                                            'model_architecture': model_arch,
                                            'n_samples': n_samples,
                                            'batch_size': batch_size,
                                            'dropout': dropout,
                                            'learning_rate': lr,
                                            'random_state': rs,
                                            'apply_augmentation': apply_aug,
                                            'n_channels': n_channels
                                        },
                                        'metrics': [val_loss, val_acc, val_preds, val_labels],
                                        'n_parameters': n_params
                                    }, model_path)
                                    log_message(f"  model_path init")
                                    result = {
                                        'config': {
                                            'model_architecture': model_arch,
                                            'n_samples': n_samples,
                                            'batch_size': batch_size,
                                            'dropout': dropout,
                                            'learning_rate': lr,
                                            'random_state': rs,
                                            'apply_augmentation': apply_aug,
                                            'n_channels': n_channels
                                        },
                                        'metrics': [val_loss, val_acc, val_preds, val_labels],
                                        'training_stats': training_stats,
                                        'inference_results': inference_results,
                                        'efficiency_metrics': efficiency_metrics,
                                        'model_path': str(model_path),
                                        'timestamp': datetime.now().isoformat()
                                    }
                                    log_message(f"  result init")
                                    all_results.append(result)
                                    completed_configs.add(config_key)
                                    
                                    with open(checkpoint_file, 'wb') as f:
                                        pickle.dump({'results': all_results, 'completed_configs': list(completed_configs)}, f)
                                    
                                    log_message(
                                        f"OK! Accuracy: {val_acc:.4f}, "
                                        f"F1: {f1score:.4f}"
                                    )
                                    
                                except Exception as e:
                                    log_message(f"ERROR: {str(e)}")
                                    traceback.print_exc()
                                    all_results.append({
                                        'config': {
                                            'model_architecture': model_arch,
                                            'n_samples': n_samples,
                                            'batch_size': batch_size,
                                            'dropout': dropout,
                                            'learning_rate': lr,
                                            'random_state': rs,
                                            'apply_augmentation': apply_aug,
                                            'n_channels': n_channels
                                        },
                                        'error': str(e)
                                    })
    
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, default=str, ensure_ascii=False)
    
    log_message(f"ЗАВЕРШЕНО! Результаты: {results_file}")
    return all_results




13. ПРИМЕР ВЫЗОВА ФУНКЦИИ run_full_model_evaluation()



In [9]:

# Пример использования:
# 1. Загрузите данные из папки с .raw файлами
# 2. Подготовьте базовые данные в нужном формате
# 3. Вызовите функцию оценки

# # Шаг 1: Загрузка данных
#data_folder = "./thicks_export"  # или путь к вашей папке с данными
data_folder = 'dataset'  # или путь к вашей папке с данными
data_list, y_all, thicknom_list = load_dataset_from_folder(data_folder)
# Convert list of matrices to numpy array
max_len = max([m.shape[0] for m in data_list])
X_all = np.array([np.pad(m, ((0, max_len - m.shape[0]), (0, 0)), mode='constant') for m in data_list])
# 

Found files: 3487
Error reading thick_378451.raw: unpack requires a buffer of 12032 bytes
Loaded: 3486 files


In [10]:
# # Шаг 2: Подготовка базовых данных
base_data = {    'X_full': X_all,     'y_full': y_all }
# 

In [20]:


# # Шаг 3: Запуск полной оценки модели
results = run_full_model_evaluation(
    base_data=base_data,
    output_dir="model_experiments",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Альтернативно, для быстрого тестирования с одним файлом:
# raw_file_path = "./thick_378786.raw"
# record = read_raw_file(raw_file_path)
# X_sample = record['data'].reshape(1, *record['data'].shape)
# y_sample = np.array([0])  # предположим, что это нормальная труба
# 
# base_data_test = {
#     'X_full': X_sample,
#     'y_full': y_sample
# }
# 
# results = run_full_model_evaluation(
#     base_data=base_data_test,
#     output_dir="test_experiments",
#     device="cpu"
# )


Загружено 10 предыдущих результатов
[2026-05-12 14:49:48] ================================================================================
[2026-05-12 14:49:48] НАЧАЛО ТЕСТИРОВАНИЯ: 7200 конфигураций
[2026-05-12 14:49:48] Архитектуры: ['OneDCNN', 'PipeCNNLSTM', 'CNNWithClassifier', 'PipeTransformer']
[2026-05-12 14:49:48] ================================================================================
[2026-05-12 14:49:48] [2/7200] PipeCNNLSTM: N=100, BS=1, DO=0.2, LR=0.001, RS=42, Aug=True
[2026-05-12 14:49:49]   Параметры модели: 2,004,770
🚀 Начало обучения на cuda
📊 Параметры модели: 2,004,770
[2026-05-12 14:49:49] ERROR: Expected more than 1 value per channel when training, got input size torch.Size([1, 128, 1])
[2026-05-12 14:49:49] [3/7200] CNNWithClassifier: N=100, BS=1, DO=0.2, LR=0.001, RS=42, Aug=True


Traceback (most recent call last):
  File "/tmp/ipykernel_86606/897302083.py", line 166, in run_full_model_evaluation
    model, history, training_stats = train_with_tracking(
  File "/tmp/ipykernel_86606/2773754283.py", line 31, in train_with_tracking
    outputs = model(data, lengths)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
  File "/tmp/ipykernel_86606/1048604469.py", line 144, in forward
    x = self.cnn(x)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/mo

[2026-05-12 14:49:49] ERROR: PipeCNNEncoder.__init__() got an unexpected keyword argument 'n_classes'
[2026-05-12 14:49:49] [4/7200] PipeTransformer: N=100, BS=1, DO=0.2, LR=0.001, RS=42, Aug=True


Traceback (most recent call last):
  File "/tmp/ipykernel_86606/897302083.py", line 145, in run_full_model_evaluation
    model = PipeCNNEncoder(
TypeError: PipeCNNEncoder.__init__() got an unexpected keyword argument 'n_classes'


[2026-05-12 14:49:49]   Параметры модели: 1,625,923
🚀 Начало обучения на cuda
📊 Параметры модели: 1,625,923
[2026-05-12 14:49:49] ERROR: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument attn_bias in method wrapper_CUDA___scaled_dot_product_efficient_attention)
[2026-05-12 14:49:49] [6/7200] PipeCNNLSTM: N=100, BS=1, DO=0.2, LR=0.001, RS=42, Aug=False
[2026-05-12 14:49:50]   Параметры модели: 2,004,770
🚀 Начало обучения на cuda
📊 Параметры модели: 2,004,770
[2026-05-12 14:49:50] ERROR: Expected more than 1 value per channel when training, got input size torch.Size([1, 128, 1])


...

Traceback (most recent call last):
  File "/tmp/ipykernel_86606/897302083.py", line 166, in run_full_model_evaluation
    model, history, training_stats = train_with_tracking(
  File "/tmp/ipykernel_86606/2773754283.py", line 31, in train_with_tracking
    outputs = model(data, lengths)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
  File "/tmp/ipykernel_86606/1048604469.py", line 144, in forward
    x = self.cnn(x)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/mo

[2026-05-12 14:50:42]   Параметры модели: 1,625,923
🚀 Начало обучения на cuda
📊 Параметры модели: 1,625,923
[2026-05-12 14:50:42] ERROR: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument attn_bias in method wrapper_CUDA___scaled_dot_product_efficient_attention)
[2026-05-12 14:50:42] [17/7200] OneDCNN: N=100, BS=1, DO=0.2, LR=0.001, RS=31, Aug=True


Traceback (most recent call last):
  File "/tmp/ipykernel_86606/897302083.py", line 166, in run_full_model_evaluation
    model, history, training_stats = train_with_tracking(
  File "/tmp/ipykernel_86606/2773754283.py", line 31, in train_with_tracking
    outputs = model(data, lengths)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
  File "/tmp/ipykernel_86606/1048604469.py", line 324, in forward
    x = self.transformer_encoder(x, src_key_padding_mask=attention_mask)  # (B, L, D)
  File "/home/strel/miniconda3/envs/torch-clean/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/strel/mi

[2026-05-12 14:50:42]   Параметры модели: 78,114
🚀 Начало обучения на cuda
📊 Параметры модели: 78,114
Epoch 01 | Train Loss: 0.6964 Acc: 59.00% | Val Loss: 0.7295 Acc: 61.19% | Time: 1.78s
Epoch 05 | Train Loss: 0.6782 Acc: 60.00% | Val Loss: 0.7042 Acc: 61.19% | Time: 1.90s


KeyboardInterrupt: 